# Load EIA Residual Fuel Oil Consumption Data from API

## Purpose
Fetches annual residual fuel oil consumption data from the EIA API and loads it directly to the bronze layer for downstream processing.

## Data Source
* **EIA International Energy API** - Residual fuel oil consumption data
  * Product ID: 54 (Residual fuel oil)
  * Activity ID: 2 (Consumption)
  * Frequency: Annual
  * Coverage: 1980-2024, all countries
  * Unit: TBPD (Thousand Barrels Per Day)

## Output
* **Table**: bronze.eia_residual_consumption_annual
* **Format**: Delta table (overwrite mode)
* **Schema**: Auto-detected from API response

## Workflow
1. **Fetch from API** - POST request with pagination (5000 records per batch) for all country IDs
2. **Create bronze schema** - Ensure workspace.bronze schema exists
3. **Load to bronze** - Write raw API data to Delta table in overwrite mode

In [0]:
import pandas as pd
import requests
from io import StringIO

base_url = "https://api.eia.gov/v2/international/data/"
api_key = "BEvLzF1LIQm3Klx8lfHIJKdAYHkxniZrifcdt0Kx"

# List of all country/region IDs to include
country_ids = [
    "ABW", "AFG", "AFRC", "AGO", "ALB", "ARE", "ARG", "ARM", "ASM", "ATA", "ATG", "AUS", "AUT", "AZE",
    "BDI", "BEL", "BEN", "BFA", "BGD", "BGR", "BHR", "BHS", "BIH", "BLR", "BLZ", "BMU", "BOL", "BRA", "BRB",
    "BRN", "BTN", "BWA", "CAF", "CAN", "CHE", "CHL", "CHN", "CIV", "CMR", "COD", "COG", "COK", "COL", "COM",
    "CPV", "CRI", "CSAM", "CSK", "CUB", "CYM", "CYP", "CZE", "DDR", "DEU", "DEUW", "DJI", "DMA", "DNK", "DOM",
    "DZA", "ECU", "EGY", "ERI", "ESH", "ESP", "EST", "ETH", "FIN", "FJI", "FLK", "FRA",
    "FRO", "GAB", "GBR", "GEO", "GHA", "GIB", "GIN", "GLP", "GMB", "GNB", "GNQ", "GRC", "GRD", "GRL", "GTM",
    "GUF", "GUM", "GUY", "HITZ", "HKG", "HND", "HRV", "HTI", "HUN", "IDN", "IND", "IRL", "IRN", "IRQ", "ISL",
    "ISR", "ITA", "JAM", "JOR", "JPN", "KAZ", "KEN", "KGZ", "KHM", "KIR", "KNA", "KOR", "KWT", "LAO", "LBN",
    "LBR", "LBY", "LCA", "LKA", "LSO", "LTU", "LUX", "LVA", "MAC", "MAR", "MDA", "MDG", "MDV", "MEX",
    "MKD", "MLI", "MLT", "MMR", "MNE", "MNG", "MOZ", "MRT", "MSR", "MTQ", "MUS", "MWI", "MYS", "MYT",
    "NAM", "NCL", "NER", "NFK", "NGA", "NIC", "NIU", "NLD", "NOR", "NPL", "NRU", "NZL", "OMN", "PAK", "PAN",
    "PER", "PHL", "PLW", "PNG", "POL", "PRI", "PRK", "PRT", "PRY", "PSE", "PYF", "QAT", "REU", "ROU", "RUS",
    "RWA", "SAU", "SDN", "SEN", "SGP", "SHN", "SJM", "SLB", "SLE", "SLV", "SMR", "SOM", "SPM", "SRB", "SSD",
    "STP", "SUR", "SVK", "SVN", "SWE", "SWZ", "SXM", "SYC", "SYR", "TCA", "TCD", "TGO", "THA", "TJK", "TKL",
    "TKM", "TLS", "TON", "TTO", "TUN", "TUR", "TUV", "TWN", "TZA", "UGA", "UKR", "URY", "USA", "UZB", "VAT",
    "VCT", "VEN", "VGB", "VIR", "VNM", "VUT", "WLF", "WSM", "XKX", "YEM", "ZAF", "ZMB", "ZWE", "WORL"
]

# Prepare the payload for ANNUAL data
payload = {
    "frequency": "annual",
    "data": [
        "value"
    ],
    "facets": {
        "countryRegionId": country_ids,
        "productId": ["54"],  # Residual fuel oil
        "activityId": ["2"]   # Consumption
    },
    "start": "1980",
    "end": "2024",
    "sort": [
        {
            "column": "period",
            "direction": "desc"
        }
    ],
    "offset": 0,
    "length": 5000
}

headers = {
    "X-Params": f'{{"api_key": "{api_key}"}}'
}

all_data = []
offset = 0
total_records = None

# Paginate through the API results
while True:
    payload["offset"] = offset
    response = requests.post(base_url, json=payload, headers=headers)
    
    if response.status_code == 200:
        result = response.json()
        data = result.get("response", {}).get("data", [])
        
        if total_records is None:
            total_records = result.get("response", {}).get("total", 0)
            print(f"Total records to fetch: {total_records}")
        
        if not data:
            break
            
        all_data.extend(data)
        offset += len(data)
        print(f"Fetched {len(all_data)} of {total_records} records")
        
        if len(all_data) >= total_records:
            break
    else:
        print(f"Error: {response.status_code}")
        print(response.text)
        break

print(f"\nTotal records fetched: {len(all_data)}")

# Convert to DataFrame
df = pd.DataFrame(all_data)
print(f"\nDataFrame shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
display(df.head())

In [0]:
# Convert pandas DataFrame to Spark DataFrame
spark_df = spark.createDataFrame(df)

# Create schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

# Write to bronze layer
spark_df.write.mode("overwrite").format("delta").saveAsTable(
    "bronze.eia_residual_consumption_annual"
)

print(f"Loaded {spark_df.count()} records to bronze.eia_residual_consumption_annual")